# Importando as libs e o DataSet

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [31]:
path_projeto = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
path_data = os.path.join(path_projeto, "data", "train.csv")
df = pd.read_csv(path_data)

# Entendendo os dados

Vamos uitlizar o info para ter uma visão geral dos dados. Esse metodo nos mostra o shape do nosso dataset, o nome de cada coluna, o tipo de cada coluna e também a quantiddade de valores não nulos em cada coluna.

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Agoras vamos utilizar o método describe para analisar alguns indices estatisticos das colunas.

In [33]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Vamos alterar o nome de cada coluna para que todas as letras possam ficar em caixa baixa.

In [34]:
func = lambda x: x.casefold()
df.columns = df.columns.map(func) 

Vamos excluir a coluna "PassengerId", pois serve apenas como indice.

In [35]:
df.drop(axis = 1, labels = 'passengerid', inplace=True)

In [36]:
df.head(2)

,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


# Limpando os dados

## pclass

In [39]:
df['pclass'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: pclass
Non-Null Count  Dtype
--------------  -----
891 non-null    int64
dtypes: int64(1)
memory usage: 7.1 KB


In [40]:
df['pclass'].unique()

array([3, 1, 2])

## name

In [14]:
df['name'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: name
Non-Null Count  Dtype 
--------------  ----- 
891 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [194]:
len(df['name'].unique())

891

## sex

In [41]:
df['sex'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: sex
Non-Null Count  Dtype 
--------------  ----- 
891 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [42]:
df['sex'].unique()

array(['male', 'female'], dtype=object)

Vamos trocar 'female' e 'male' para 0 e 1 respectivamente.

In [43]:
df['sex'].replace(['male', 'female'], [1, 0], inplace=True)

In [44]:
df.head(3)

,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## age

In [46]:
df['age'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: age
Non-Null Count  Dtype  
--------------  -----  
714 non-null    float64
dtypes: float64(1)
memory usage: 7.1 KB


A coluna "age" possui alguns valores vazios que precisam ser tratados.

Resolvemos preencher os valores vazios com base na média de idade do grupo no qual a pessoa se encaixa, grupo esse formado com base no titulo da pessoa, para facilitar esse preenchimento nós criamos uma coluna title que contem o titulo da pessoa. Então se uma pessoa possui um valor vazio na coluna idade e na coluna title essa pessoa possui o title igual a Mr, logo a idade dessa pessoa será preenchida com base na média de todas as pessoas que possuem o title igual a Mr.

Criando a coluna title.

In [47]:
title = df["name"].apply(func = lambda t: t.split(',')[1].split('.')[0].strip())
df.loc[:,'title'] = title

In [48]:
df.head(3)

,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,title
0,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,C85,C,Mrs
2,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss


Preenchendo os valores vazios.

In [49]:
# Criando um dict com a media de idade de cada grupo
media_por_title = df.groupby(by='title', sort=False)['age'].mean().round(0).to_dict()

# Criando um novo df
df2 = df.copy()

# Percorrendo e torcanco os nan pela media do grupo onde a pessoa se encaixa
for pos in df2[ df2['age'].isna() ].index:
    df2.loc[pos, 'age'] = media_por_title[df2['title'][pos]]

In [50]:
# Não existe mais valores vazios
df2['age'].count()

891

## sibsp

In [52]:
df['sibsp'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: sibsp
Non-Null Count  Dtype
--------------  -----
891 non-null    int64
dtypes: int64(1)
memory usage: 7.1 KB


In [53]:
df['sibsp'].unique()

array([1, 0, 3, 4, 2, 5, 8])

## parch

In [28]:
df['parch'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: parch
Non-Null Count  Dtype
--------------  -----
891 non-null    int64
dtypes: int64(1)
memory usage: 7.1 KB


In [54]:
df['parch'].unique()

array([0, 1, 2, 5, 3, 4, 6])

## ticket

In [55]:
df['ticket'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: ticket
Non-Null Count  Dtype 
--------------  ----- 
891 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [56]:
len(df['ticket'].unique())

681

## fare

In [57]:
df['fare'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: fare
Non-Null Count  Dtype  
--------------  -----  
891 non-null    float64
dtypes: float64(1)
memory usage: 7.1 KB


In [58]:
len(df['fare'].unique())

248

## cabin

In [59]:
df['cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [60]:
len(df['cabin'].unique())

148

## embarked

Seguimos o site (https://www.encyclopedia-titanica.org/titanic-survivor/martha-evelyn-stone.html) para preencher os valores vazios dessa coluna.

In [61]:
df['embarked'].replace(np.nan, "S", inplace=True)

In [63]:
df['embarked'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: embarked
Non-Null Count  Dtype 
--------------  ----- 
891 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [64]:
df['embarked'].unique()

array(['S', 'C', 'Q'], dtype=object)

# Analise exploratória uni e bivariada